In [44]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import f1_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/tania/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [45]:
stop_words = stopwords.words('spanish')
print(f"Total stopwords: {len(stop_words)}")
print(stop_words[:20])

Total stopwords: 313
['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo']


In [46]:
# Cargar datos preprocesados
df_train = pd.read_csv('../data/train_clean.csv')
df_test  = pd.read_csv('../data/test_clean.csv')

# Separar features y etiquetas
X_train = df_train['MESSAGE_CLEAN']
y_train = df_train['IS_IRONIC']
X_test  = df_test['MESSAGE_CLEAN']
y_test  = df_test['IS_IRONIC']

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución train: {y_train.value_counts().to_dict()}")

Train: (7197,) | Test: (1800,)
Distribución train: {0: 4797, 1: 2400}


In [47]:
# Vectorización TF-IDF con eliminación de stopwords en español
# La eliminación de stopwords para ML tradicional con TF-IDF es práctica
# estándar documentada en detección de sarcasmo (Manoleasa et al., 2022;
# Šandor & Bagić Babac, 2023), aunque aplicada en inglés. Se adopta el
# mismo principio para español dado que TF-IDF no captura contexto
# gramatical, a diferencia de los modelos DL de este estudio.

tfidf = TfidfVectorizer(
    stop_words=stopwords.words('spanish'),
    max_features=10000, # Pandey & Singh (2023); Šandor & Bagić Babac (2023)
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Matriz train: {X_train_tfidf.shape}")
print(f"Matriz test:  {X_test_tfidf.shape}")

Matriz train: (7197, 10000)
Matriz test:  (1800, 10000)


In [48]:
from sklearn.pipeline import Pipeline

# Pipeline LR
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stopwords.words('spanish'),
        max_features=10000,
    )),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced' ))
])

# Pipeline RF
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stopwords.words('spanish'),
        max_features=10000
    )),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced' ))
])

# Grillas centradas en valores de Pandey & Singh (2023)
param_grid_lr = {
    'tfidf__ngram_range': [(1,1), (1,2), (2,3)],
    'clf__C': [0.1, 1.0, 10],
    'clf__solver': ['lbfgs', 'liblinear']
}

param_grid_rf = {
    'tfidf__ngram_range': [(1,1), (1,2), (2,3)],
    'clf__n_estimators': [100, 200],  # valor base: 100 (Pandey & Singh, 2023)
    'clf__max_depth': [None, 10, 20], # valor base: None (Pandey & Singh, 2023)
    'clf__min_samples_split': [2, 5]  # valor base: 2 (Pandey & Singh, 2023)
}

print("Pipelines y grillas definidos")

Pipelines y grillas definidos


In [49]:
# Stratified K-Fold para mantener proporción de clases en cada fold
# ante el desbalance 2:1 del dataset
# Anan et al. y Fatima et al. aplican validación cruzada estratificada de 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV para LR
# Se usa Pipeline para evitar data leakage:
# el TF-IDF se reajusta solo con datos de entrenamiento en cada fold
print("Buscando hiperparámetros para LR...")
search_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
search_lr.fit(X_train, y_train)
print(f"Mejores parámetros LR: {search_lr.best_params_}")
print(f"Mejor F1-Macro LR (CV): {search_lr.best_score_:.4f}")

Buscando hiperparámetros para LR...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Mejores parámetros LR: {'clf__C': 1.0, 'clf__solver': 'liblinear', 'tfidf__ngram_range': (1, 2)}
Mejor F1-Macro LR (CV): 0.6542


In [50]:
# GridSearchCV para RF
print("Buscando hiperparámetros para RF...")
search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
search_rf.fit(X_train, y_train)
print(f"Mejores parámetros RF: {search_rf.best_params_}")
print(f"Mejor F1-Macro RF (CV): {search_rf.best_score_:.4f}")

Buscando hiperparámetros para RF...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Mejores parámetros RF: {'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 100, 'tfidf__ngram_range': (1, 1)}
Mejor F1-Macro RF (CV): 0.6380


In [51]:
# Evaluación final en test con los mejores modelos encontrados
# Se usa el mejor estimador encontrado por GridSearchCV
best_lr = search_lr.best_estimator_
best_rf = search_rf.best_estimator_

# Predicciones
y_pred_lr = best_lr.predict(X_test)
y_pred_rf = best_rf.predict(X_test)

# Resultados
print("=== Regresión Logística ===")
print(f"F1-Macro test: {f1_score(y_test, y_pred_lr, average='macro'):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['No irónico', 'Irónico']))

print("\n=== Random Forest ===")
print(f"F1-Macro test: {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['No irónico', 'Irónico']))

=== Regresión Logística ===
F1-Macro test: 0.6486
              precision    recall  f1-score   support

  No irónico       0.77      0.75      0.76      1201
     Irónico       0.52      0.55      0.54       599

    accuracy                           0.68      1800
   macro avg       0.65      0.65      0.65      1800
weighted avg       0.69      0.68      0.69      1800


=== Random Forest ===
F1-Macro test: 0.6380
              precision    recall  f1-score   support

  No irónico       0.75      0.83      0.79      1201
     Irónico       0.56      0.44      0.49       599

    accuracy                           0.70      1800
   macro avg       0.65      0.63      0.64      1800
weighted avg       0.68      0.70      0.69      1800



In [52]:
import joblib

joblib.dump(best_lr, '../data/modelo_lr.pkl')
joblib.dump(best_rf, '../data/modelo_rf.pkl')
print("Modelos guardados: modelo_lr.pkl | modelo_rf.pkl")

Modelos guardados: modelo_lr.pkl | modelo_rf.pkl
